In [1]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
from nltk.sentiment import SentimentIntensityAnalyzer
import requests
import re
import tqdm
import io
import os
from PyPDF2 import PdfReader

# List of FNO Stocks

In [2]:
fno_list = pd.read_csv('fno_stocks_list.csv')

drop_words = [' Limited', ' Ltd', ' Industries', 'The ', ' (India)', ' (india)',' Enterprises',' Enterprise', ' Company', ' Laboratories', ' Corporation']
fno_list['Stock'] = fno_list['Stock Name']

for word in drop_words:
    fno_list['Stock'] = fno_list['Stock'].map(lambda x: x.replace(word, ''))
    
fno_list['Stock'] = fno_list['Stock'].map(lambda x: x.lower())
fno_list['Symbol'] = fno_list['Symbol'].map(lambda x: x.lower())

symbols = fno_list['Symbol'].values
stocks = fno_list['Stock'].values
filter = sorted(list(set(list(symbols) + list(stocks))))

# Read Announcements from donwloaded CSV

In [7]:
nse_df = pd.read_csv('nse_data.csv')
nse_df['SYMBOL'] = nse_df['SYMBOL'].apply(lambda x: x.lower())
nse_df1 = nse_df[nse_df['SYMBOL'].isin(symbols)]
nse_df1 = nse_df1.reset_index(drop=True)

skip_subject = ['Loss of Share Certificates', 'Loss/Duplicate-Share Certificate-XBRL', 
               'Change in Directors/ Key Managerial Personnel/ Auditor/ Compliance Officer/ Share Transfer Agent',
                'Analysts/Institutional Investor Meet/Con. Call Updates', 'Investor Presentation',
               'ESOP/ESOS/ESPS','Board Meeting Intimation','Alteration Of Capital and Fund Raising-XBRL', 'Record Date',
               'Trading Window', 'Change in Director(s)','Cessation']
less_oimp_subjects = ['Credit Rating','Resignation', 'Copy of Newspaper Publication','Change in Management','Appointment',
                     'Outcome of Board Meeting', 'Trading Window-XBRL','Sale or disposal-XBRL','Committee Meeting Updates',
                     'Certificate under SEBI (Depositories and Participants) Regulations, 2018','Monthly Business Updates']

important_subject = ['Reply to Clarification Sought', 'Press Release', 'Updates', 'News Verification',
                    'Memorandum of Understanding/Agreements','Acquisition-XBRL','Acquisition', 'Amalgamation OR Merger-XBRL',
                    'ISD for Buyback-Tender Offer', 'Amalgamation/Merger',]
nse_imp = nse_df1[nse_df1['SUBJECT'].isin(important_subject)]
nse_imp.loc[:,'DISSEMINATION'] = pd.to_datetime(nse_imp['DISSEMINATION'])

In [8]:
start_date = pd.to_datetime('2024-03-14')
nse_imp = nse_imp[nse_imp['DISSEMINATION'] > start_date]
nse_imp

,SYMBOL,COMPANY NAME,SUBJECT,DETAILS,BROADCAST DATE/TIME,RECEIPT,DISSEMINATION,DIFFERENCE,ATTACHMENT
2,hdfcbank,HDFC Bank Limited,Updates,HDFC Bank Limited has informed the Exchange re...,20-Mar-2024 22:57:44,2024-03-20 22:57:44,2024-03-20 22:57:48,00:00:04,https://nsearchives.nseindia.com/corporate/HDF...
7,polycab,Polycab India Limited,Updates,Polycab India Limited has informed the Exchang...,20-Mar-2024 21:52:25,2024-03-20 21:52:25,2024-03-20 21:52:32,00:00:07,https://nsearchives.nseindia.com/corporate/POL...
9,lalpathlab,Dr. Lal Path Labs Ltd.,Updates,Dr. Lal Path Labs Ltd. has informed the Exchan...,20-Mar-2024 21:29:35,2024-03-20 21:29:35,2024-03-20 21:29:40,00:00:05,https://nsearchives.nseindia.com/corporate/LAL...
11,icicibank,ICICI Bank Limited,Updates,ICICI Bank Limited has informed the Exchange r...,20-Mar-2024 20:40:21,2024-03-20 20:40:21,2024-03-20 20:40:27,00:00:06,https://nsearchives.nseindia.com/corporate/ICI...
15,tvsmotor,TVS Motor Company Limited,Acquisition,TVS Motor Company Limited has informed the Exc...,20-Mar-2024 20:14:34,2024-03-20 20:14:34,2024-03-20 20:14:40,00:00:06,https://nsearchives.nseindia.com/corporate/TVS...
16,tvsmotor,TVS Motor Company Limited,Acquisition,TVS Motor Company Limited has informed the Exc...,20-Mar-2024 20:08:56,2024-03-20 20:08:56,2024-03-20 20:09:07,00:00:11,https://nsearchives.nseindia.com/corporate/TVS...
37,icicipruli,ICICI Prudential Life Insurance Company Limited,Updates,ICICI Prudential Life Insurance Company Limite...,20-Mar-2024 18:42:08,2024-03-20 18:42:08,2024-03-20 18:42:14,00:00:06,https://nsearchives.nseindia.com/corporate/ICI...
44,crompton,Crompton Greaves Consumer Electricals Limited,Press Release,Crompton Greaves Consumer Electricals Limited ...,20-Mar-2024 18:15:24,2024-03-20 18:15:24,2024-03-20 18:15:32,00:00:08,https://nsearchives.nseindia.com/corporate/CRO...
58,deepakntr,Deepak Nitrite Limited,Updates,Deepak Nitrite Limited has informed the Exchan...,20-Mar-2024 17:40:12,2024-03-20 17:40:12,2024-03-20 17:40:19,00:00:07,https://nsearchives.nseindia.com/corporate/DEE...
69,abb,ABB India Limited,Updates,ABB India Limited has informed the Exchange re...,20-Mar-2024 17:24:11,2024-03-20 17:24:11,2024-03-20 17:24:21,00:00:10,https://nsearchives.nseindia.com/corporate/ABB...


In [9]:
len(nse_imp)

29

In [10]:
nse_imp.to_csv('nse_imp.csv')

In [36]:
# Filter Stocks by Market Cap


# Filter Stocks by Liquidity


#Filter Stocks by FNO Volume Spurt


#Best Case is Low Market Cap, Decent Liquidity and Volume Spurt

In [18]:
nse_df1['SUBJECT'].unique()

array(['Loss/Duplicate-Share Certificate-XBRL',
       'Loss of Share Certificates', 'Copy of Newspaper Publication',
       'Updates',
       'Analysts/Institutional Investor Meet/Con. Call Updates',
       'Alteration Of Capital and Fund Raising-XBRL',
       'Allotment of Securities', 'Notice Of Shareholders Meetings-XBRL',
       'ESOP/ESOS/ESPS', 'Shareholders meeting',
       'Change in Directors/ Key Managerial Personnel/ Auditor/ Compliance Officer/ Share Transfer Agent',
       'Change in Management', 'Acquisition-XBRL', 'Press Release',
       'Investor Presentation', 'Appointment',
       'Memorandum of Understanding/Agreements',
       'Outcome of Board Meeting', 'Trading Window-XBRL', 'Credit Rating',
       'Board Meeting Intimation', 'Record Date', 'Resignation',
       'Trading Window', 'Change in Director(s)', 'Sale or disposal-XBRL',
       'News Verification', 'Acquisition', 'Amalgamation OR Merger-XBRL',
       'Committee Meeting Updates',
       'Certificate under 

In [16]:
headers = {'User-Agent': 'Mozilla/5.0 (X11; Windows; Windows x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/103.0.5060.114 Safari/537.36'}
response = requests.get(url=nse_df1.loc[0,"ATTACHMENT"], headers=headers, timeout=120)
on_fly_mem_obj = io.BytesIO(response.content)
pdf_file = PdfReader(on_fly_mem_obj)

In [20]:
page = pdf_file.pages[0]
text = page.extract_text()
text

'~ (~~~I) fclflle\'5 \n(~"fRc!iRqiT~ -~~) \nGAIL (India) Limited \n(A Government of India Undertaking-A Maharatna Company) \nND/GAIL/SECTT /2024 \n1. Listing Compliance \nNational Stock Exchange of India Limited \nExchange Plaza, 5th Floor, Plot No. C/1, \nG Block, Bandra-Kurla Complex, \nBandra (East), Mumbai -400051 \nScrip Code: GAIL-EQ 2. Listing Compliance \nBSE Limited, Tf(\'f~. \n16 ~ cl>flTT ~ \n~ ~-110066. \'+fR<l \nGAIL BHAWAN. \n16 BHIKAIJI CAMA PLACE \nNEW DELHl-110066, INDIA \nq;\'r.,/PHONE : +9111 26182955 \n~/FAX: +9111 26185941 \nt-i\'rc;r/E-mail: info@gail.co.in \n04.03.2024 \nFloor 1, Phiroze Jeejeebhoy Towers, \nDalal Street, \nMumbai-400001 \nScrip Code: 532155 \nSub: Schedule of Analyst/ Institutional Investor Meet/ Conference \nDear Sir/Madam , \nThis is to inform that in terms of Regulation 30 of the SEBI (Listing Obligations and \nDisclosure Requirements) Regulations , 2015, schedule of Analyst/ Institutiona l Investor \nMeet /Conference is proposed to be attend

In [21]:
nse_df1.loc[0,"SUBJECT"]

'Analysts/Institutional Investor Meet/Con. Call Updates'

In [24]:
for i in nse_df1.index:
    print(nse_df1.loc[i,"DETAILS"], "\n")

GAIL (India) Limited has informed the Exchange about Schedule of meet 

Container Corporation Of India Limited has informed the Exchange regarding Cessation of Mr A K CHANDRA as Non- Executive Director of the company w.e.f. February 29, 2024. 

OIL & NATURAL GAS CORPORATION LIMITED has informed the Exchange about Change in Directors/ Key Managerial Personnel/ Auditor/ Compliance Officer/ Share Transfer Agent 

Oil & Natural Gas Corporation Limited has informed the Exchange regarding 'Change in Senior Management'. 

HDFC Bank Limited has informed the Exchange about Loss of share certificates 

HDFC Bank Limited has informed the Exchange about Loss of Share Certificates 

Pidilite Industries Limited has informed the Exchange about Schedule of meet 

SBI Cards and Payment Services Limited has informed the Exchange about Schedule of meet 

SBI Cards and Payment Services Limited has informed the Exchange about Schedule of meet 

Power Grid Corporation of India Limited has informed the Excha

In [11]:
news_df[news_df['Tags']!='']

,Date,Headlines,Description,Source,Source_Link,Tags
24,"09:02 PM, 02 Mar 2024",pm flags off 1st crude oil tanker from ongc's ...,"the ongc's kg block kg-dwn-98/2, which started...",moneycontrol,https://www.moneycontrol.com/news/business/pm-...,ongc
74,"04:45 PM, 02 Mar 2024","mukesh ambani: anant and radhika, ‘yeh rab ne ...",anant ambani & radhika merchant pre-wedding: t...,moneycontrol,https://www.moneycontrol.com/news/videos/india...,reliance
90,"02:48 PM, 02 Mar 2024",remembering sumant moolgaokar who kickstarted ...,legend has it that jrd tata handpicked sumant ...,moneycontrol,https://www.moneycontrol.com/news/trends/featu...,idea
101,"02:00 PM, 02 Mar 2024",swachhatapukare: weaving prosperity from a sou...,explore the story of swachhatapukare with gaur...,moneycontrol,https://www.moneycontrol.com/news/videos/speci...,au small finance bank
116,"12:57 PM, 02 Mar 2024",record highs for sensex & nifty on glitch-free...,"indian headline indices, s&p bse sensex and ni...",economic times,https://economictimes.indiatimes.com/markets/s...,"itc, tata motors, tata steel"
124,"12:30 PM, 02 Mar 2024","tata steel shares jump over 4% on block deal, ...",tata steel shares have delivered over 40% retu...,economic times,https://economictimes.indiatimes.com/markets/s...,tata steel
125,"12:26 PM, 02 Mar 2024",live: saturday trading session | nifty trades ...,saturday trading session: nifty hits new high ...,moneycontrol,https://www.moneycontrol.com/news/videos/busin...,"adani, tata motors, tata steel"
131,"11:40 AM, 02 Mar 2024",aurobindo pharma soars 4% on usfda nod to manu...,aurobindo pharma share price today: fingolimod...,moneycontrol,https://www.moneycontrol.com/news/business/mar...,aurobindo pharma
140,"11:04 AM, 02 Mar 2024",info edge shares fall 3% after google removes ...,the company in its filing to the exchanges sai...,economic times,https://economictimes.indiatimes.com/markets/s...,info edge
145,"10:35 AM, 02 Mar 2024","cci must take action against google, says info...","google delisted five of info edge’s apps, incl...",moneycontrol,https://www.moneycontrol.com/news/technology/c...,"info edge, naukri"


# Filter Articles for FNO Stocks

In [9]:
#Add Stock as a tag in a new column if it's mentioned in the description
news_df['Tags'] = ''
for i in news_df.index:
    desc = news_df.loc[i,'Description']
    tags = [s for s in filter if re.search(r'\b{}\b'.format(re.escape(s)), desc)]

    #tags are under a single string separated by ',' and not an iterable of strings
    
    tags = ', '.join(tags)
    if len(tags) > 0:
        news_df.loc[i,'Tags'] = tags
    else:
        continue

In [ ]:
#Drop rows which have on tags i.e. no mention of stock of interest
news_df = news_df[news_df['Tags']!= '']
news_df = news_df.reset_index(drop=True)

In [ ]:
#If more than 3 stocks mentioned, it is a generic article and not particular to a stock. Ignore?

# Get Detailed News Article

In [7]:
def get_article_text(source, source_link):
    if source == 'moneycontrol':
        try:
            article_response = requests.get(source_link, headers=request_headers)
            article_html = BeautifulSoup(article_response.text, 'html')
            article_html = article_html.find('div', {'class': 'content_wrapper arti-flow'})
            paragraph = article_html.find_all('p')
            
            paragraph_text = ''
            for i in range(len(paragraph)):
                if paragraph[i].get('class') == ['benefitText']:
                    break
                paragraph_text += paragraph[i].text.strip()
            return paragraph_text
            
        except:
            return "Error ocurred"
        
    if source == 'bloomberg quint':
        try:
            article_response = requests.get(source_link, headers=request_headers)
            article_html = BeautifulSoup(article_response.text, 'html')
            paragraph = article_html.find_all('p')
            
            paragraph_text = ''
            for i in range(len(paragraph)):
                if paragraph[i].get('class') == ['benefitText']:
                    break
                paragraph_text += paragraph[i].text.strip()
            return paragraph_text
            
        except:
            return "Error ocurred"
            

    if source == 'economic times':
        try:
            article_response = requests.get(source_link, headers=request_headers)
            article_html = BeautifulSoup(article_response.text, 'html')
            paragraph = article_html.find('article')
            paragraph_text = paragraph.text.strip().split('(You can now subscribe to our ETMarkets WhatsApp channel)')[0]
    
            return paragraph_text
            
        except:
            return "Error ocurred"

    if source == 'the hindu business':
        try:
            article_response = requests.get(source_link, headers=request_headers)
            article_html = BeautifulSoup(article_response.text, 'html')
            article_html = article_html.find('div', {'itemprop': 'articleBody'})
            paragraph = article_html.find_all('p')
            
            paragraph_text = ''
            for i in range(len(paragraph)):
                paragraph_text += paragraph[i].text.strip()
            paragraph_text = paragraph_text.split('COMMents')[0]
            return paragraph_text
            
        except:
            return "Error ocurred"
        
    if source == 'zee business':
        try:
            article_response = requests.get(source_link, headers=request_headers)
            article_html = BeautifulSoup(article_response.text, 'html')
            article_html = article_html.find('div', {'class': 'field-item even'})
            paragraph = article_html.find_all('p')
    
            paragraph_text = ''
            for i in range(len(paragraph)):
                    paragraph_text += paragraph[i].text.strip()
            return paragraph_text
            
        except:
            return "Error ocurred"
        
    if source == 'finshots':
        try:  
            article_response = requests.get(source_link, headers=request_headers)
            article_html = BeautifulSoup(article_response.text, 'html')
            article_html = article_html.find('div', {'class': 'post-content'})
            paragraph = article_html.find_all('p')
    
            paragraph_text = ''
            for i in range(len(paragraph)):
                    paragraph_text += paragraph[i].text.strip()
            paragraph_text = paragraph_text.split("Don't forget to share this story on WhatsApp")[0]
            return paragraph_text
        except:
            return "Error ocurred"
        
    else:
        return 'New Website. Create a rapper for {}'.format(source)

In [8]:
news_df['Complete_Article'] = ''

for i in news_df.index:
    news_df.loc[i, 'Complete_Article'] = get_article_text(news_df.loc[i, 'Source'], news_df.loc[i, 'Source_Link'])

In [25]:
#Filtering out errors and paid articles
news_df = news_df[news_df['Complete_Article']!='Error ocurred']
news_df = news_df[news_df['Complete_Article']!='']

In [26]:
#Exporting
news_df.to_pickle("news.pkl")

 # Pass Description through ChatGPT and get an output on price outlook.

# Debugging

In [27]:
news_df[news_df['Complete_Article']=='Error ocurred']

,Date,Headlines,Description,Source,Source_Link,Tags,Complete_Article


In [19]:
total_text = ''
for value in news_df['Complete_Article'].values:
    total_text += value

In [22]:
len(total_text.split())

11975

In [23]:
11975*22

263450

In [28]:
news_df

,Date,Headlines,Description,Source,Source_Link,Tags,Complete_Article
0,"10:58 PM, 01 Mar 2024",trade setup for saturday: 15 things to know be...,"a long build-up was seen in 76 stocks, which i...",moneycontrol,https://www.moneycontrol.com/news/business/mar...,"national aluminium, sail, tvs motor",After a robust performance by the Nifty 50 on ...
1,"10:43 PM, 01 Mar 2024","hdfc bank, 3 others settle case with sebi; pay...","hdfc bank, hsbc, citi bank, and deutsche bank ...",bloomberg quint,https://www.ndtvprofit.com/business/hdfc-bank-...,hdfc bank,"HDFC Bank, HSBC, Citi Bank and Deutsche Bank A..."
2,"09:52 PM, 01 Mar 2024",gujarat delegation led by cm patel to visit ra...,a state government release said gujarat assemb...,moneycontrol,https://www.moneycontrol.com/news/india/gujara...,balkrishna,Gujarat Chief Minister Bhupendra Patel and his...
3,"08:10 PM, 01 Mar 2024","paytm, paytm payments bank to discontinue inte...",the move assumes significance as paytm payment...,the hindu business,https://www.thehindu.com/business/paytm-paytm-...,persistent,"Amid RBI’s action on its associate firm, One 9..."
4,"07:54 PM, 01 Mar 2024","maruti, hyundai, tata motors register robust s...","mahindra & mahindra, toyota kirloskar motor an...",bloomberg quint,https://www.ndtvprofit.com/business/maruti-hyu...,mahindra & mahindra,"Leading automakers Maruti Suzuki, Hyundai and ..."
5,"07:02 PM, 01 Mar 2024","bse rejig: jfs added to bse large cap index, t...","container corp. of india, mphasis ltd., and vo...",bloomberg quint,https://www.ndtvprofit.com/markets/bse-rejig-j...,"mphasis, voltas",The Bombay Stock Exchange announced its period...
6,"05:26 PM, 01 Mar 2024",stock market update: stocks that hit 52-week l...,"dr. reddys, sun pharma, hcl tech, infosys a...",economic times,https://economictimes.indiatimes.com/markets/s...,"britannia, infosys","Getty ImagesNEW DELHI: GPT Healthcare, Stamped..."
9,"02:07 PM, 01 Mar 2024",piramal enterprises & avanti feeds nagaraj she...,"nagaraj shetti, technical & derivative analyst...",economic times,https://economictimes.indiatimes.com/markets/e...,"pel, piramal",ETMarkets.comRelatedTechnical Stock Pick: Mult...
10,"02:01 PM, 01 Mar 2024",buy manappuram finance; target of rs 215: geojit,geojit is bullish on manappuram finance has re...,moneycontrol,https://www.moneycontrol.com/news/business/sto...,"manappuram, manappuram finance",Geojit's research report on Manappuram Finance...
11,"01:54 PM, 01 Mar 2024",ministry of defence signs five major capital a...,"the contracts include one worth rs 5,249.72 cr...",moneycontrol,https://www.moneycontrol.com/news/business/min...,"hal, hindustan aeronautics",The Ministry of Defence (MoD) has signed five ...


In [12]:
nse_df = pd.read_csv("nse_announcements.csv")
nse_df

,SYMBOL,COMPANY NAME,SUBJECT,DETAILS,BROADCAST DATE/TIME,RECEIPT,DISSEMINATION,DIFFERENCE,ATTACHMENT
0,ROHLTD,Royal Orchid Hotels Limited,Updates,Royal Orchid Hotels Limited has informed the E...,02-Mar-2024 23:50:45,2024-03-02 23:50:45,02-Mar-2024 23:50:54,00:00:09,https://nsearchives.nseindia.com/corporate/ROH...
1,CENTUM,Centum Electronics Limited,Loss of Share Certificates,Centum Electronics Limited has informed the Ex...,02-Mar-2024 23:17:31,2024-03-02 23:17:31,02-Mar-2024 23:17:36,00:00:05,https://nsearchives.nseindia.com/corporate/CEN...
2,CENTUM,Centum Electronics Limited,Loss/Duplicate-Share Certificate-XBRL,CENTUM ELECTRONICS LIMITED has informed the Ex...,02-Mar-2024 23:15:58,2024-03-02 23:15:58,02-Mar-2024 23:16:04,00:00:06,https://nsearchives.nseindia.com/corporate/xbr...
3,PHOENIXLTD,The Phoenix Mills Limited,Updates,The Phoenix Mills Limited has informed the Exc...,02-Mar-2024 22:59:39,2024-03-02 22:59:39,02-Mar-2024 22:59:45,00:00:06,https://nsearchives.nseindia.com/corporate/PHO...
4,GOLDENTOBC,Golden Tobacco Limited,Corporate Insolvency Resolution Process,Golden Tobacco Limited has informed the Exchan...,02-Mar-2024 22:56:16,2024-03-02 22:56:16,02-Mar-2024 22:56:21,00:00:05,https://nsearchives.nseindia.com/corporate/GOL...
...,...,...,...,...,...,...,...,...,...
185,TITAN,Titan Company Limited,Acquisition-XBRL,TITAN COMPANY LIMITED has informed the Exchang...,02-Mar-2024 07:05:31,2024-03-02 07:05:31,02-Mar-2024 07:05:39,00:00:08,https://nsearchives.nseindia.com/corporate/xbr...
186,REFEX,Refex Industries Limited,Shareholders meeting,Pursuant to Regulation 30 and 44 of the SEBI (...,02-Mar-2024 01:41:27,2024-03-02 01:41:27,02-Mar-2024 01:41:33,00:00:06,https://nsearchives.nseindia.com/corporate/REF...
187,INDUSINDBK,IndusInd Bank Limited,Alteration Of Capital and Fund Raising-XBRL,INDUSIND BANK LIMITED has informed the Exchang...,02-Mar-2024 00:40:03,2024-03-02 00:40:03,02-Mar-2024 00:40:10,00:00:07,https://nsearchives.nseindia.com/corporate/xbr...
188,INDUSINDBK,IndusInd Bank Limited,Allotment of Securities,Indusind Bank Limited has informed the Exchang...,02-Mar-2024 00:38:43,2024-03-02 00:38:43,02-Mar-2024 00:38:48,00:00:05,https://nsearchives.nseindia.com/corporate/IND...


In [15]:
nse_df["SYMBOL"] = nse_df["SYMBOL"].apply(lambda x: x.lower())
nse_fno_df = nse_df[nse_df["SYMBOL"].isin(symbols)]

In [17]:
nse_fno_df.to_csv('nse_fno.csv')